# Treino com GPU no Colab (Parte A: A5 ResNet e A6 CNN + LSTM)

Este notebook roda **no Colab** (Ambiente de execução → Alterar o tipo → **GPU**). Ele:
1. baixa o código do GitHub e os dados da CODE-15% (partes 0 a 3) para o disco da máquina do Colab;
2. confere o ambiente reproduzindo o A3 congelado na validação (macro AUROC tem que dar **0,9892**);
3. treina o A6 (CNN + LSTM) e o A5 (ResNet pequena), mudando **só a arquitetura** em relação ao A3;
4. ao fim de cada treino, baixa um `.zip` com os resultados para o seu computador (pasta Downloads). O Google Drive é opcional (`USAR_DRIVE`), desligado porque usaria o Drive da conta logada no Colab.

Só treino e validação: o grupo teste é cortado na carga e nunca é lido. Técnicas fora das aulas (ResNet, LSTM).
Os dados ficam no disco do Colab, não no Drive: ler o HDF5 do Drive seria lento. Eles somem quando a sessão fecha.

## 1. GPU e pasta de resultados

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import tensorflow as tf, keras
print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))
assert tf.config.list_physical_devices("GPU"), "Sem GPU: Ambiente de execução -> Alterar o tipo de ambiente -> GPU"

In [ ]:
USAR_DRIVE = False  # True = também copia os resultados para o Google Drive DA CONTA LOGADA NO COLAB (pode pedir a senha)
import os, time
RODADA = time.strftime("%Y%m%d_%H%M")
SAIDA = f"/content/saida_{RODADA}"          # resultados desta rodada (no disco do Colab; no fim vira um .zip para baixar)
os.makedirs(SAIDA, exist_ok=True)
if USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = f"/content/drive/MyDrive/ecg_colab/{RODADA}"
    os.makedirs(DRIVE, exist_ok=True)
print("resultados desta rodada:", SAIDA, "| Drive:", USAR_DRIVE)

## 2. Código (GitHub)

In [ ]:
import os
REPO = "/content/repo"
if not os.path.exists(REPO):
    !git clone -q https://github.com/releventhal/Entrega-final-Insper-AI.git {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}
!git log -1 --oneline
!pip -q install h5py

## 3. Dados: `exams.csv` e partes 0 a 3 (~2,7 GB cada zip)

In [ ]:
import time, zipfile, urllib.request, shutil
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
ZENODO = "https://zenodo.org/records/4916206/files"
PARTES = [0, 1, 2, 3]
DADOS = Path(REPO) / "code15"
DADOS.mkdir(exist_ok=True)

def baixar(url, destino, conexoes=16):
    """Baixa em pedaços paralelos (HTTP Range), retomando conexões que caem; confere o tamanho total (mesma função do projeto_ecg.ipynb)."""
    with urllib.request.urlopen(urllib.request.Request(url, headers={"Range": "bytes=0-0"})) as r:
        total = int(r.headers["Content-Range"].split("/")[1])
    with open(destino, "wb") as f:
        f.truncate(total)
    passo = -(-total // conexoes)
    baixado = [0] * conexoes
    def pedaco(i):
        ini, fim = i * passo, min(total, (i + 1) * passo) - 1
        for _ in range(100):
            if baixado[i] >= fim - ini + 1:
                return
            try:
                pos = ini + baixado[i]
                req = urllib.request.Request(url, headers={"Range": f"bytes={pos}-{fim}"})
                with urllib.request.urlopen(req, timeout=60) as r, open(destino, "r+b") as f:
                    f.seek(pos)
                    while bloco := r.read(1 << 20):
                        f.write(bloco); baixado[i] += len(bloco)
            except Exception as e:
                print(f"pedaço {i}: {e!r}, retomando"); time.sleep(3)
        raise RuntimeError(f"pedaço {i} incompleto")
    with ThreadPoolExecutor(conexoes) as ex:
        list(ex.map(pedaco, range(conexoes)))
    assert sum(baixado) == total, "download incompleto"

if not (DADOS / "exams.csv").exists():
    urllib.request.urlretrieve(f"{ZENODO}/exams.csv?download=1", DADOS / "exams.csv")
for p in PARTES:
    h5 = DADOS / f"exams_part{p}.hdf5"
    if h5.exists():
        continue
    z = DADOS / f"exams_part{p}.zip"
    t0 = time.time()
    baixar(f"{ZENODO}/exams_part{p}.zip?download=1", z)
    zipfile.ZipFile(z).extractall(DADOS)
    z.unlink()
    print(f"parte {p}: {(time.time() - t0) / 60:.1f} min")
!df -h /content | tail -1
!ls -la {DADOS}

## 4. Lista de exames zerados (a mesma regra do A3: só os com as 12 derivações zeradas saem)

In [ ]:
import numpy as np, ecg_dados as E
arq = E.DADOS / "exames_zerados_p0-3.npy"
if not arq.exists():
    z = E.listar_zerados([0, 1, 2, 3])
    np.save(arq, z)
z = np.load(arq)
print("com alguma derivação zerada:", len(z), "| com as 12 zeradas:", int((z[:, 3] == 12).sum()), "(esperado: 411 e 172, CLAUDE.md seção 4)")
assert len(z) == 411 and (z[:, 3] == 12).sum() == 172

## 5. Conferência do ambiente: o A3 congelado na validação tem que dar macro AUROC 0,9892

In [ ]:
!python -u avaliar_externo.py --json congelado_candidato_A3.json --conjunto validacao_A3 --boot 50 2>&1 | grep --line-buffered -v -i "warning\|absl\|oneDNN" | tail -25

## 6. Tempo por lote (bench)
Só mede velocidade (15 lotes). Se a leitura do HDF5 for muito maior que o cálculo, o gargalo é o disco, não a GPU.

In [ ]:
!python -u arquiteturas.py bench A6 2>&1 | grep --line-buffered -v -i "warning\|absl\|oneDNN"
!python -u arquiteturas.py bench A5 2>&1 | grep --line-buffered -v -i "warning\|absl\|oneDNN"

## 7. Treinos (cada célula baixa um .zip ao terminar)

In [ ]:
import shutil, glob
def salvar(nome):
    """Copia os resultados do experimento para a pasta da rodada (e para o Drive, se USAR_DRIVE) e baixa um .zip."""
    for f in glob.glob(f"resultados/{nome}_*") + ["resultados/experimentos.csv"] + glob.glob(f"modelos/{nome}_melhor.keras"):
        shutil.copy(f, SAIDA)
        if USAR_DRIVE:
            shutil.copy(f, DRIVE)
    print("salvo:", sorted(os.listdir(SAIDA)))
    zipado = shutil.make_archive(f"/content/resultados_{RODADA}", "zip", SAIDA)
    from google.colab import files
    files.download(zipado)   # o navegador baixa o .zip (vai para a pasta Downloads do seu computador)

In [ ]:
!python -u arquiteturas.py treinar A6 --max-min 240 --max-epocas 20 2>&1 | grep --line-buffered -v -i "warning\|absl\|oneDNN"
salvar("A6")

In [ ]:
!python -u arquiteturas.py treinar A5 --max-min 240 --max-epocas 20 2>&1 | grep --line-buffered -v -i "warning\|absl\|oneDNN"
salvar("A5")

## 8. Comparação na validação completa (8.158 exames): A3 × A5 × A6

In [ ]:
import pandas as pd
df = pd.read_csv("resultados/experimentos.csv")
sel = df[df.experimento.str.match(r"^A[356] ") & ~df.experimento.str.contains(r"\|") & (df.rotulo == "MACRO")]
print(sel[["experimento", "auroc", "auprc", "f1", "sensibilidade", "precisao", "epocas", "melhor_epoca", "minutos_treino"]].round(4).to_string(index=False))
por_rotulo = df[df.experimento.str.match(r"^A[356] ") & ~df.experimento.str.contains(r"\|") & (df.rotulo != "MACRO")]
print(por_rotulo.pivot(index="rotulo", columns="experimento", values="auprc").round(3))